In [ ]:
# A décommenter à la première exécution si jupyter se plaint de ne pas trouver oracledb
#!pip install --upgrade oracledb

In [ ]:
# Compléter ici les imports dont vous avez besoin, ne pas modifier ceux déjà présents
import getpass
from os import getenv
import pandas as pd
import oracledb
import warnings
import matplotlib.pyplot as plt

In [ ]:
# Nécessaire pour éviter les problèmes de session
class Connexion(object):
    def __init__(self, login, password):
        self.conn = oracledb.connect(
            user=login,
            password=password,
            host="oracle.iut-orsay.fr",
            port=1521,
            sid="etudom",
        )
        self.conn.autocommit = False

    def __enter__(self):
        self.conn.autocommit = False
        return self.conn

    def __exit__(self, *args):
        self.conn.close()

In [ ]:
# La fonction ci-dessous est à utiliser pour exécuter une requête et stocker les résultats dans un dataframe Pandas sans afficher d’alerte.
# Vous pouvez vous en inspirer pour créer vos propres fonctions.
def requete_vers_dataframe(connexion_data, requete, valeurs = None):
    with Connexion(login=connexion_data['login'], password=connexion_data['password']) as connexion:
        warnings.simplefilter(action='ignore', category=UserWarning)
        if valeurs is not None:
            df = pd.read_sql(requete, connexion, params=valeurs)
        else:
            df = pd.read_sql(requete, connexion)
        warnings.simplefilter("always") 
        return df

def requete_insert(connexion_data, requete, valeurs = None):
    with Connexion(login=connexion_data['login'], password=connexion_data['password']) as connexion:
        warnings.simplefilter(action='ignore', category=UserWarning)
        warnings.simplefilter("always")
        if valeurs is not None:
            df = pd.read_sql(requete, connexion, params=valeurs)
        else:
            df = pd.read_sql(requete, connexion)
        warnings.simplefilter("always")

In [ ]:
# Saisir ci-dessous la plateforme qui vous a été attribuée. Cela correspond au nomPlateforme dans la table PLATEFORME de la base de données
# Par exemple NOM_PLATEFORME = "Switch"
NOM_PLATEFORME = "Commodore C64/128/MAX"
# Saisir ci-dessous le login court de la base utilisée pour votre carnet
SCHEMA = '"AMETIN"'

# TABLEAU DE BORD PLATEFORME

## Partie consultation des données

In [ ]:
# On demande à l'utilisateur son login et mot de passe pour pouvoir accéder à la base de données
if getenv("DB_LOGIN") is None:
    login = input("Login : ")
else:
    login = getenv("DB_LOGIN")
if getenv("DB_PASS") is None:
    password = getpass.getpass("Mot de passe : ")
else:
    password = getenv("DB_PASS")
conn = {'login': login, 'password': password}

In [ ]:
# On vérifie que l'utilisateur est bien connecté à la base de données, que le schéma est bon, et qu'on trouve la bonne édition des JO
data = requete_vers_dataframe(conn, f"SELECT * FROM {SCHEMA}.PLATEFORME WHERE nomPlateforme LIKE (:libelle)",{"libelle":NOM_PLATEFORME})
id_plateforme = int(data.IDPLATEFORME.iloc[0])
print(f"Identifiant de la plateforme : {id_plateforme}")

### Statistiques de base

#### Nombre de jeux publiés sur cette plateforme

In [ ]:
NbJeu = requete_vers_dataframe(conn,
f"""SELECT Count(idJeu) as Jeux FROM {SCHEMA}.PLATEFORME
NATURAL JOIN {SCHEMA}.DATESORTIE
NATURAL JOIN {SCHEMA}.JEU
where idplateforme={id_plateforme}
"""
)
print("Nombre de jeux sur la " + NOM_PLATEFORME +" : "+ str(NbJeu["JEUX"][0]))

#### Nombre de compagnies ayant développé sur cette plateforme

In [ ]:
NbCompagnie = requete_vers_dataframe(conn,
f"""SELECT Count(idCompagnie) as nbCompagnie FROM {SCHEMA}.PLATEFORME
NATURAL JOIN {SCHEMA}.DATESORTIE
NATURAL JOIN {SCHEMA}.JEU
NATURAL JOIN {SCHEMA}.COMPAGNIEJEU
NATURAL JOIN {SCHEMA}.COMPAGNIE  
WHERE idplateforme = {id_plateforme} 
"""
)

print("Nombre de compagnies ayant developpé,publié,... sur la " + NOM_PLATEFORME +" : "+ str(NbCompagnie["NBCOMPAGNIE"][0]))

### Jeux

#### Répartition des jeux par année

In [ ]:
# Calculer avec une requête et afficher le nombre de jeux publiés par année sur cette plateforme
data = requete_vers_dataframe(conn,f"Select Round(EXTRACT(YEAR FROM DateSortie)) as année, count(idjeu) as nbjeu From {SCHEMA}.Plateforme Natural join DateSortie Natural Join Jeu where nomPlateforme Like (:libelle) Group by EXTRACT(YEAR FROM DateSortie) order by EXTRACT(YEAR FROM DateSortie)",{"libelle":NOM_PLATEFORME})
print(data)



In [ ]:
liste = list(data["NBJEU"]) 

#etendre l'histogramme
plt.figure(figsize=(12, 6))
#création de l'histogramme (valeurs)
plt.bar(data["ANNÉE"], data["NBJEU"], color="mediumpurple", edgecolor="black")

plt.title("Nombre de jeux par année",color="black")
plt.xlabel("Année",color="black")
plt.ylabel("Nombre de jeux", color="black")
plt.xticks(rotation=45,color="crimson")
plt.yticks(rotation=45,color="crimson")
plt.grid(axis='y', linestyle='-.', alpha=0.6)
plt.grid(axis='x', linestyle='-.', alpha=0.6)
plt.tight_layout()
plt.show()

#### Top 10 des jeux les mieux notés par les utilisateurs sur cette plateforme

In [ ]:
noteU = requete_vers_dataframe(conn,f"Select titrejeu, ScoreJeu From {SCHEMA}.Plateforme Natural join DateSortie Natural Join Jeu where nomPlateforme Like (:libelle) and ScoreJeu is not null order by ScoreJeu Desc",{"libelle":NOM_PLATEFORME})
NoteUtilisateur = [noteU["TITREJEU"][i] +" : "+ str(noteU["SCOREJEU"][i]) for i in range(10)]
for i in range (0,10):
    print(str(i+1)+"  "+NoteUtilisateur[i])

### Compagnies

#### Tableau de répartition des compagnies ayant développé sur cette plateforme par pays

#### Répartition des jeux par catégorie

In [ ]:
# Récupérer les données, les transformer si nécessaire, puis afficher sous forme de tableau

dataCompagnie = requete_vers_dataframe(conn,
f"""SELECT PaysCompagnie,Count(idCompagnie) as nbCompagnie FROM {SCHEMA}.PLATEFORME
NATURAL JOIN {SCHEMA}.DATESORTIE
NATURAL JOIN {SCHEMA}.JEU
NATURAL JOIN {SCHEMA}.COMPAGNIEJEU
NATURAL JOIN {SCHEMA}.COMPAGNIE  
WHERE idplateforme = 15 and PaysCompagnie is not null and estDeveloppeur = 1
Group by PaysCompagnie
"""
)
liste1=[]  
for elt in [dataCompagnie["PAYSCOMPAGNIE"][i] for i in range(len(dataCompagnie))]:
    x=requete_vers_dataframe(conn,f"Select NomPays From PaysIso where Code_Numerique={elt}")
    liste1.append(x["NOMPAYS"][0])

# Affichage
    
liste2 = [[liste1[i],str(dataCompagnie["NBCOMPAGNIE"][i])] for i in range (len(dataCompagnie))]

fig, ax = plt.subplots()
ax.axis('off')

table = ax.table(
cellText=liste2,
colLabels=["Pays", "Compagnie"],
cellLoc='center',
loc='center'
)

table.scale(1, 1.5)

#style pour l'entête
for col in range(2):
    table[(0, col)].set_facecolor("mediumvioletred")
    table[(0, col)].get_text().set_color("white")
    table[(0, col)].get_text().set_fontweight("bold")

# remplit les doubles cellules en vert

for i in range(len(liste2)):
    for j in range(2):
        table[(i+1, j)].set_facecolor("lightgreen")

plt.title("Nombre de compagnie par pays", pad=20, color="black")
plt.show()

#### Graphique : top 25 des compagnies ayant développé sur cette plateforme les mieux notées en moyenne par les utilisateurs d'IGDB

In [ ]:
bestCompagnieDev = requete_vers_dataframe(conn,
f"""SELECT NomCompagnie,(Sum(scoreJeu)/Count(ScoreJeu)) as score FROM {SCHEMA}.PLATEFORME
NATURAL JOIN {SCHEMA}.DATESORTIE
NATURAL JOIN {SCHEMA}.JEU
NATURAL JOIN {SCHEMA}.COMPAGNIEJEU
NATURAL JOIN {SCHEMA}.COMPAGNIE  
WHERE idplateforme = 15 and estDeveloppeur = 1 and ScoreJeu is Not Null
Group by NomCompagnie
order by (Sum(scoreJeu)/Count(ScoreJeu)) Desc
"""
)

for i in range(25):
     print(str(i+1) + "  " + bestCompagnieDev["NOMCOMPAGNIE"][i] + " : " + str(bestCompagnieDev["SCORE"][i]))

#### Graphiques : top 25 des compagnies ayant le plus développé, top 25 des compagnies ayant le plus publié, en nombre de jeux, sur cette plateforme

In [ ]:
print("Top 25 des compagnies de developpement : \n")

for i in range(25):
     print(str(i+1) + "  " + bestCompagnieDev["NOMCOMPAGNIE"][i] + " : " + str(bestCompagnieDev["SCORE"][i]))

print("\n")

bestCompagniePub = requete_vers_dataframe(conn,
f"""SELECT NomCompagnie,Count(idjeu) as score FROM {SCHEMA}.PLATEFORME
NATURAL JOIN {SCHEMA}.DATESORTIE
NATURAL JOIN {SCHEMA}.JEU
NATURAL JOIN {SCHEMA}.COMPAGNIEJEU
NATURAL JOIN {SCHEMA}.COMPAGNIE  
WHERE idplateforme = 15 and estPublieur = 1
Group by NomCompagnie
order by Count(idjeu) Desc
"""
)
print("Top 25 des compagnies publieuses : \n")
for i in range(25):
     print(str(i+1) + "  " + bestCompagniePub["NOMCOMPAGNIE"][i] + " : " + str(bestCompagniePub["SCORE"][i]))    

## Partie modification des données

In [ ]:
NbJeu = requete_vers_dataframe(conn,f"Select Count(idJeu) as nbJeu From {SCHEMA}.Plateforme Natural join DateSortie Natural Join Jeu where nomPlateforme Like (:libelle)",{"libelle":NOM_PLATEFORME})

noteC = requete_vers_dataframe(conn,f"Select distinct(titrejeu), ScoreIGDB From {SCHEMA}.Plateforme Natural join DateSortie Natural Join Jeu where nomPlateforme Like (:libelle) and ScoreIGDB is not null order by ScoreIGDB Desc",{"libelle":NOM_PLATEFORME})
noteCU = requete_vers_dataframe(conn,f"Select distinct(titrejeu), (ScoreJeu + ScoreIGDB)/2 as ScoreTotal From {SCHEMA}.Plateforme Natural join DateSortie Natural Join Jeu where nomPlateforme Like (:libelle) and ScoreJeu is not null and ScoreIGDB is not Null order by (ScoreJeu + ScoreIGDB)/2 Desc",{"libelle":NOM_PLATEFORME})

NoteCritique = [noteC["TITREJEU"][i] +" : "+ str(noteC["SCOREIGDB"][i]) for i in range(10)]
NoteCombine = [noteCU["TITREJEU"][i] +" : "+ str(noteCU["SCORETOTAL"][i]) for i in range(10)]

print("Nombre de jeu de la " + NOM_PLATEFORME + " : " +str(NbJeu["NBJEU"][0]) + " jeuxn")
print("Top 10 des Jeux de la "+NOM_PLATEFORME+" selon les utilisateurs\n")
for i in range (0,10):
    print(str(i+1)+"  "+NoteUtilisateur[i])

print("nTop 10 des Jeux de la "+NOM_PLATEFORME+" selon les critiques\n")  
for i in range (0,10):
    print(str(i+1)+"  "+NoteCritique[i])

print("nTop 10 des Jeux de la "+NOM_PLATEFORME+"n")  
for i in range (0,10):
    print(str(i+1)+"  "+NoteCombine[i])    

### Ajout d'une nouvelle compagnie

Ajoutez à la base une nouvelle compagnie ayant pour nom *Binôme XXX* où XXX est votre numéro de binôme.

In [ ]:
with Connexion(login=conn['login'], password=conn['password']) as connexion:  # Démarre une nouvelle connexion
    connexion.begin()  # Démarre une transaction
    curseur = connexion.cursor()  # On prépare un curseur pour exécuter une requête
    requete = f""" INSERT INTO {SCHEMA}.Compagnie Values(
        (Select COALESCE(Max(idcompagnie),0)+1 From {SCHEMA}.Compagnie),
        :nom,
        :description,
        :pays,
        SYSDATE,
        SYSDATE,
        NULL
        )""" 
    donnees = {'nom':"47",'description':"La compagnie avec une aura infinie", 'pays':"France"}  # Les données à insérer pour cette requête
    curseur.execute(requete, donnees)  # On demande au serveur d'exécuter la requête
    connexion.commit()  # On valide la transaction


### Ajout d'un nouveau jeu

Ajouter le nouveau jeu "BDWarrior 2025" développé et publié par la compagnie crée précédemment. Pour le moment, les scores et nombre de notes sont vides, et vous pouvez choisir les autres propriétés à votre goût.

In [ ]:
# Ajouter le jeu ici
requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.Jeu
    VALUES (
        (SELECT COALESCE(MAX(idJeu), 0) + 1 FROM {SCHEMA}.Jeu),
        :titre,
        :titreAlt,
        :Lore,
        :Resume,
        NULL,
        NULL,
        NULL,        
        NULL,
        NULL,
        NULL,
        :tempsNormal,
        :tempsRapide,
        :tempsComplet,
        :NombreTemps,
        'release',
        SYSDATE,
        NULL,
        NULL,
        NULL,
        1
    )
""")

with Connexion(login=conn['login'], password=conn['password']) as connexion:  # Démarre une nouvelle connexion
    connexion.begin()  # Démarre une transaction
    curseur = connexion.cursor()  # On prépare un curseur pour exécuter une requête
    requete = f""" INSERT INTO {SCHEMA}.Compagnie Values(
        (Select COALESCE(Max(idcompagnie),0)+1 From {SCHEMA}.Compagnie),
        :nom,
        :description,
        :pays,
        SYSDATE,
        SYSDATE,
        NULL
        )""" 
    donnees = {'titre':"BDWarrior 2025",'titreAlt':"BDWarrior 2025", 'Lore':"L''histoire de la BDD",'Resume':"Combattre l''administrateur de la base de données à l''aide de requêtes SQL", 'tempsNormal':1000,'tempsRapide':700,'tempsComplet':1500,'NombreTemps':19092}  # Les données à insérer pour cette requête
    curseur.execute(requete, donnees)  # On demande au serveur d'exécuter la requête
    connexion.commit()  # On valide la transaction


# Faire ensuite les liens avec les tables COMPAGNIE, THEME, CATEGORIEJEU, REGION, MOTCLE...
requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.Motcle
    SELECT
        COALESCE(MAX(idMotCle), 0) + 1,
        'Base de donnée',
    FROM {SCHEMA}.Motcle
""")

requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.Motcle
    SELECT
        COALESCE(MAX(idMotCle), 0) + 1,
        'SQL',
    FROM {SCHEMA}.Motcle
""")

requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.MotcleJeu
    Values(
        Select idMotCle from {SCHEMA}.MOTCLE where nomMotcle="SQL",
        Select Max(idJeu) From {SCHEMA}.Jeu
    )
""")

requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.MotcleJeu
    Values(
        Select idMotCle from {SCHEMA}.MOTCLE where nomMotcle="Base de donnée",
        Select Max(idJeu) From {SCHEMA}.Jeu
    )
""")

requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.Genre
    Select
        COALESCE(Max(idGenre),0)+1,
        'RPG'
    From {SCHEMA}.Genre    
""")


requete_insert(conn, f"""
    INSERT INTO {SCHEMA}.GenreJeu
    Values(
        Select idgenre from {SCHEMA}.Genre where nomGenre="RPG",
        Select Max(idjeu) From {SCHEMA}.GenreJeu
    )
""")



### Ajout d'une nouvelle date de sortie

Le jeu est sorti mondialement le 28/05/2025 sur la plateforme qui vous a été assignée.

In [ ]:
# Ajouter la sortie ici

### Affichage

In [ ]:
# On affiche la fiche détaillée des sorties du jeu (avec DETAIL_SORTIES)

### Première vérification

On vérifie maintenant que le nombre de jeux sur la plateforme reflète l'ajout que l'on vient de faire.

### Evaluation du jeu

Votre jeu est un immense succès, il obtient une note parfaite tant chez les utilisateurs que chez les critiques. Mettez à jour la base pour refléter cela.

In [ ]:
# Faire la mise à jour des notes

### Seconde vérification

On regarde maintenant le top 10 des jeux les mieux notés sur les trois critères sur la plateforme

In [ ]:
# Afficher les nouveaux top 10 ici

### Nettoyage

Annulez maintenant vos modifications

In [ ]:
# Nettoyage de la base ici (avec des requêtes DELETE)